<a href="https://colab.research.google.com/github/chetools/CHE4071_Spring2026/blob/main/pH_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [2]:
def netH(pH):
    return jnp.where(pH<7, 10**(-pH), -10**(-14+pH))

def pH(netH):
    return jnp.squeeze(jnp.where(netH>0, -jnp.log10(netH), 14.+jnp.log10(-netH)))

dpH_netH = jax.grad(pH)

In [9]:
def make_step(t0):
    def step(t):
        return jnp.where(t>t0, 1, 0.)
    return step

In [31]:
V = 10.  #L
pHA, pHB = 1., 13.
pH_setpoint = 6.5
Kc = 1e7



In [32]:
def dpHdt(t, pH, u):
    qin, pHin = u
    netH_err = netH(pH_setpoint)-netH(pH)
    qA = jnp.where(netH_err>0, Kc*jnp.abs(netH_err), 0.)
    qB = jnp.where(netH_err<0, Kc*jnp.abs(netH_err), 0.)

    dnetH_dt = (qin*netH(pHin) + qA*netH(pHA) + qB*netH(pHB) - (qin+qA+qB)*netH(pH))/V
    return dpH_netH(pH) * dnetH_dt

In [33]:
pHin_step = make_step(100.)

#disturbance in inlet pH
def dpHdt2(t, pH, u):
    qin = u
    pHin = 6.5 + 1.5*pHin_step(t)
    netH_err = netH(pH_setpoint)-netH(pH)
    qA = jnp.where(netH_err>0, Kc*jnp.abs(netH_err), 0.)
    qB = jnp.where(netH_err<0, Kc*jnp.abs(netH_err), 0.)
    dnetH_dt = (qin*netH(pHin) + qA*netH(pHA) + qB*netH(pHB) - (qin+qA+qB)*netH(pH))/V
    return dpH_netH(pH) * dnetH_dt

In [34]:
u = jnp.array([10.])

In [35]:
tend = 500.
tplot = np.linspace(0,tend,100)
res=sp.integrate.solve_ivp(lambda t, pH: dpHdt2(t, pH, u), (0, tend), jnp.array([1.]), method='Radau', dense_output=True)
pHplot = res.sol(tplot)[0]

In [36]:
fig=make_subplots()
fig.add_scatter(x=tplot, y=pHplot)
fig.update_layout(width=600, height=400)